# 1. Cleaning the Telefónica data

**Question:** what did Telefónica actually deliver, and what had to be fixed before it can be trusted?

The delivery is a set of mobility CSVs (one folder per month, Jan 2023 - Aug 2025, three files each) and nine small tourism files per month. `prisa.ingest` converts it into checked tables:

```
uv run python -m prisa.ingest data/raw/telefonica --out data/processed
```

Every table has a written contract in `prisa/schemas.py` (columns, types, keys, allowed values) and is checked against it when written.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prisa import store, figures
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
con = store.connect()

## What we found in the data, beyond the spec

| Finding | Consequence |
|---|---|
| Barri codes have 9 digits with the leading zero (`080190101`) | `prisa.geo` uses 9-digit barri ids; 5-digit INE codes for municipalities |
| The last two digits restart in each district (Bon Pastor = `0903`, officially barri 59) | `data/reference/barri_crosswalk.csv` maps codes to official numbers; checked by name for all 73 |
| M3 (barri <-> municipality) repeats a key several times with different counts | Rows are **summed**: the sums track M2's municipality totals, deduplicating doesn't |
| M3 captures less than M2 reports for the same municipality pairs | Finer cells fall under the 15-trip threshold more often. We never add M2 and M3 together |
| Trips < 15 are published as 0 | Every trip row carries `censored`; totals are lower bounds |
| Tourism files use `;` and write age bands as `1829` | Handled in `prisa.ingest.tourism` |
| Groups < 15 people are pooled into `NA` remainder rows, even in key columns | Split off and measured in the ingest report |

In [ ]:
print((store.PROCESSED / 'ingest_report.md').read_text())

## Checking the M3 sub-row decision on one month

If the repeated M3 rows are real sub-rows, summing them over barris should reproduce M2's municipality totals for trips touching Barcelona (minus what censoring hides).

In [ ]:
check = con.execute('''
WITH m2 AS (SELECT day, segment, origin_zone_id o, dest_zone_id d, trips FROM trip_flows
            WHERE source = 'muni_muni' AND year = 2024 AND month(day) = 5
              AND (origin_zone_id = '08019' OR dest_zone_id = '08019')),
     m3 AS (SELECT day, segment,
                   CASE WHEN length(origin_zone_id) = 9 THEN '08019' ELSE origin_zone_id END o,
                   CASE WHEN length(dest_zone_id) = 9 THEN '08019' ELSE dest_zone_id END d, sum(trips) trips
            FROM trip_flows WHERE source = 'barri_muni' AND year = 2024 AND month(day) = 5 GROUP BY ALL)
SELECT corr(m2.trips, m3.trips) r, sum(m3.trips) / sum(m2.trips) captured
FROM m2 JOIN m3 USING (day, segment, o, d) WHERE m2.trips > 500''').df()
check

## The two daily tables everything else starts from

`prisa.features` builds `barri_day` (73 barris x 971 days: trips in/out by segment, visitors present by segment and type) and `city_day` (city-wide visitors and trips).

In [ ]:
bd = store.read('barri_day'); cd = store.read('city_day')
print(bd.shape, cd.shape)
bd.describe().T.round(0)